# CIFAR10-C Robustness Evaluation

This notebook evaluates trained models on CIFAR10-C, a benchmark dataset for testing robustness to common corruptions.

## What is CIFAR10-C?

CIFAR10-C contains 15 corruption types at 5 severity levels:
- **Noise**: gaussian_noise, shot_noise, impulse_noise
- **Blur**: defocus_blur, glass_blur, motion_blur, zoom_blur
- **Weather**: snow, frost, fog, brightness
- **Digital**: contrast, elastic_transform, pixelate, jpeg_compression

## Setup

First, download CIFAR10-C from: https://zenodo.org/record/2535967

Extract it to your data folder (e.g., `data/CIFAR-10-C/`)

In [ ]:
import sys, os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if (project_root / 'src').exists():
    sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import CIFAR10C, get_cifar10c_loader, get_all_cifar10c_loaders
from src.utils.training import calculate_accuracy, get_normalizer, evaluate_cifar10c, load_weights
from src.utils.visualization import *

## 1. Verify CIFAR10-C Data

First, let's check if the data is available and visualize some corrupted images.

In [ ]:
# Set path to CIFAR10-C
CIFAR10C_PATH = DATA_PATH / 'CIFAR-10-C'

# Check if data exists
if not CIFAR10C_PATH.exists():
    print(f"⚠️  CIFAR10-C not found at: {CIFAR10C_PATH}")
    print("\nDownload from: https://zenodo.org/record/2535967")
    print("Extract to:", CIFAR10C_PATH)
else:
    print(f"✓ CIFAR10-C found at: {CIFAR10C_PATH}")
    
    # List available corruption files
    npy_files = list(CIFAR10C_PATH.glob('*.npy'))
    print(f"\nFound {len(npy_files)} .npy files")
    print("\nAvailable corruptions:")
    for f in sorted(npy_files):
        if f.name != 'labels.npy':
            print(f"  - {f.stem}")

## 2. Visualize Corruptions

Let's look at examples of different corruption types and severities.

In [ ]:
def visualize_corruption(corruption_type, image_idx=0):
    """Visualize a single image across all 5 severity levels."""
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    
    # Load clean CIFAR10 image for comparison
    from src.utils.datasets import CIFAR10
    clean_dataset = CIFAR10(train=False, transform=None)
    clean_img, label = clean_dataset[image_idx]
    
    # Show clean image
    axes[0].imshow(clean_img)
    axes[0].set_title(f'Clean\n{clean_dataset.classes[label]}')
    axes[0].axis('off')
    
    # Show corrupted versions
    for severity in range(1, 6):
        dataset = CIFAR10C(CIFAR10C_PATH, corruption=corruption_type, severity=severity, transform=None)
        img, _ = dataset[image_idx]
        
        axes[severity].imshow(img)
        axes[severity].set_title(f'Severity {severity}')
        axes[severity].axis('off')
    
    fig.suptitle(f'Corruption: {corruption_type}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize a few different corruptions
corruptions_to_show = ['gaussian_noise', 'motion_blur', 'snow', 'contrast']

for corruption in corruptions_to_show:
    if (CIFAR10C_PATH / f"{corruption}.npy").exists():
        visualize_corruption(corruption, image_idx=42)

## 3. Load Your Trained Models

Load the models you want to evaluate.

In [ ]:
# Import your models
from src.models.architectures.RestNet18 import ResNet18
from src.models.architectures.ScatNet18 import ScatNet18
from src.models.architectures.ScatPoolRestNet18 import ScatPoolResNet18

# Initialize models
models_to_eval = {}

# Example: Load your baseline ResNet18
# Adjust the experiment_name and model_name to match your saved checkpoints
try:
    baseline = ResNet18(num_classes=10).to(DEVICE)
    load_weights(baseline, experiment_name='baseline', model_name='ResNet18', device=DEVICE)
    models_to_eval['ResNet18_baseline'] = baseline
    print("✓ Loaded ResNet18 baseline")
except Exception as e:
    print(f"⚠️  Could not load ResNet18 baseline: {e}")

# Add more models as needed
# try:
#     scatnet = ScatNet18(num_classes=10, J=4, L=8).to(DEVICE)
#     load_weights(scatnet, experiment_name='your_exp', model_name='ScatNet18', device=DEVICE)
#     models_to_eval['ScatNet18'] = scatnet
#     print("✓ Loaded ScatNet18")
# except Exception as e:
#     print(f"⚠️  Could not load ScatNet18: {e}")

print(f"\nTotal models loaded: {len(models_to_eval)}")

## 4. Quick Test on Single Corruption

Test a model on one corruption type to verify everything works.

In [ ]:
if models_to_eval:
    model_name = list(models_to_eval.keys())[0]
    model = models_to_eval[model_name]
    
    # Test on gaussian noise at different severities
    print(f"Testing {model_name} on gaussian_noise:")
    
    for severity in [1, 3, 5]:
        loader = get_cifar10c_loader(CIFAR10C_PATH, 'gaussian_noise', severity=severity, batch_size=128)
        acc = calculate_accuracy(model, loader, DEVICE)
        print(f"  Severity {severity}: {acc:.2f}%")
else:
    print("No models loaded. Please load at least one model first.")

## 5. Full CIFAR10-C Evaluation

Evaluate on all 15 corruptions and 5 severity levels.

In [ ]:
# Store results for all models
all_results = {}

for model_name, model in models_to_eval.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    results = evaluate_cifar10c(
        model=model,
        cifar10c_root=CIFAR10C_PATH,
        device=DEVICE,
        batch_size=128
    )
    
    all_results[model_name] = results

print("\n" + "="*60)
print("Evaluation complete!")
print("="*60)

## 6. Visualize Results

Create visualizations comparing model performance across corruptions.

In [ ]:
def plot_corruption_comparison(results_dict):
    """Plot accuracy across all corruptions for each model."""
    corruptions = CIFAR10C.CORRUPTIONS
    
    # Calculate average accuracy across all severities for each corruption
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(corruptions))
    width = 0.8 / len(results_dict)
    
    for i, (model_name, results) in enumerate(results_dict.items()):
        avg_accs = []
        for corruption in corruptions:
            if corruption in results['accuracies']:
                avg_acc = np.mean([results['accuracies'][corruption][s] for s in [1,2,3,4,5]])
                avg_accs.append(avg_acc)
            else:
                avg_accs.append(0)
        
        ax.bar(x + i * width, avg_accs, width, label=model_name, alpha=0.8)
    
    ax.set_xlabel('Corruption Type', fontsize=12)
    ax.set_ylabel('Average Accuracy (%)', fontsize=12)
    ax.set_title('Model Robustness Across CIFAR10-C Corruptions\n(Average across all severity levels)', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * (len(results_dict) - 1) / 2)
    ax.set_xticklabels(corruptions, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

if all_results:
    plot_corruption_comparison(all_results)

In [ ]:
def plot_severity_heatmap(results, model_name):
    """Plot heatmap showing accuracy across corruptions and severities."""
    corruptions = CIFAR10C.CORRUPTIONS
    severities = [1, 2, 3, 4, 5]
    
    # Create accuracy matrix
    acc_matrix = np.zeros((len(corruptions), len(severities)))
    for i, corruption in enumerate(corruptions):
        if corruption in results['accuracies']:
            for j, severity in enumerate(severities):
                acc_matrix[i, j] = results['accuracies'][corruption][severity]
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(8, 10))
    im = ax.imshow(acc_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)
    
    # Set ticks and labels
    ax.set_xticks(np.arange(len(severities)))
    ax.set_yticks(np.arange(len(corruptions)))
    ax.set_xticklabels([f'Sev {s}' for s in severities])
    ax.set_yticklabels(corruptions)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Accuracy (%)', rotation=270, labelpad=20)
    
    # Add text annotations
    for i in range(len(corruptions)):
        for j in range(len(severities)):
            text = ax.text(j, i, f'{acc_matrix[i, j]:.1f}',
                          ha="center", va="center", color="black", fontsize=8)
    
    ax.set_title(f'{model_name}\nAccuracy Across Corruptions and Severities', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Plot heatmap for each model
for model_name, results in all_results.items():
    plot_severity_heatmap(results, model_name)

In [ ]:
def plot_mean_corruption_error(results_dict):
    """Plot mean corruption error for each model."""
    model_names = list(results_dict.keys())
    mces = [results_dict[name]['mean_corruption_error'] for name in model_names]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(model_names, mces, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'][:len(model_names)])
    
    ax.set_ylabel('Mean Corruption Error (%)', fontsize=12)
    ax.set_title('Model Robustness Comparison\n(Lower is Better)', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, mce in zip(bars, mces):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{mce:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

if all_results:
    plot_mean_corruption_error(all_results)

## 7. Summary Statistics

Print detailed summary of results.

In [ ]:
def print_summary_table(results_dict):
    """Print a summary table of model performance."""
    print("\n" + "="*80)
    print("CIFAR10-C EVALUATION SUMMARY")
    print("="*80)
    
    for model_name, results in results_dict.items():
        print(f"\nModel: {model_name}")
        print("-" * 80)
        print(f"Mean Corruption Error: {results['mean_corruption_error']:.2f}%")
        
        # Find best and worst corruptions
        corruption_avg_accs = {}
        for corruption in results['accuracies']:
            avg_acc = np.mean([results['accuracies'][corruption][s] for s in [1,2,3,4,5]])
            corruption_avg_accs[corruption] = avg_acc
        
        best_corruption = max(corruption_avg_accs, key=corruption_avg_accs.get)
        worst_corruption = min(corruption_avg_accs, key=corruption_avg_accs.get)
        
        print(f"\nBest Performance: {best_corruption} ({corruption_avg_accs[best_corruption]:.2f}% avg)")
        print(f"Worst Performance: {worst_corruption} ({corruption_avg_accs[worst_corruption]:.2f}% avg)")
        
        # Category breakdown
        categories = {
            'Noise': ['gaussian_noise', 'shot_noise', 'impulse_noise'],
            'Blur': ['defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur'],
            'Weather': ['snow', 'frost', 'fog', 'brightness'],
            'Digital': ['contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
        }
        
        print("\nCategory Breakdown (average accuracy):")
        for category, corruptions in categories.items():
            accs = [corruption_avg_accs[c] for c in corruptions if c in corruption_avg_accs]
            if accs:
                print(f"  {category:12s}: {np.mean(accs):.2f}%")

if all_results:
    print_summary_table(all_results)

## 8. Save Results

Save the evaluation results for later analysis.

In [ ]:
import pickle
from datetime import datetime

if all_results:
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = STATS_PATH / f"cifar10c_results_{timestamp}.pkl"
    
    with open(results_file, 'wb') as f:
        pickle.dump(all_results, f)
    
    print(f"Results saved to: {results_file}")
else:
    print("No results to save.")